# 05주차 4번 과제 — 하이퍼파라미터 튜닝 비교

**Dataset**: Breast Cancer Wisconsin (종양 진단)
**Model**: RandomForestClassifier

## 실험 설계

| | Experiment A | Experiment B |
|---|---|---|
| 방법 | **Grid Search Only** | **Random Search → Grid Search** |
| 탐색 방식 | 넓은 격자 전수 탐색 | Stage1: 광역 랜덤 → Stage2: 정밀 격자 |
| 목표 | 동일한 파라미터 공간을 GridSearch만으로 커버 | RandomSearch로 유망 구간을 먼저 좁힌 후 정밀 탐색 |

> 두 실험의 **최종 정확도, 총 학습 횟수(fits), 소요 시간**을 비교한다.

In [ ]:
%matplotlib inline
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint
from itertools import product

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
)
from sklearn.metrics import accuracy_score, classification_report

SEED = 42
CV   = 5
np.random.seed(SEED)
print("sklearn ready")

## 데이터셋 준비

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
class_names = data.target_names  # ['malignant', 'benign']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"샘플: {len(X)}  |  특성: {X.shape[1]}")
print(f"Train/Test: {len(X_train)}/{len(X_test)}")
print(f"클래스: {dict(zip(class_names, np.bincount(y)))}")

## 로거 정의

In [ ]:
class SearchLogger:
    """각 탐색 단계의 시간·학습 횟수·성능을 기록"""

    def __init__(self, experiment_name: str):
        self.name   = experiment_name
        self.stages = []
        self._exp_start = None

    def start_experiment(self):
        self._exp_start = time.time()
        print(f"\n{'='*65}")
        print(f"  Experiment: {self.name}")
        print(f"{'='*65}")

    def log_stage(self, stage_name: str, search_obj, n_fits: int, elapsed: float):
        test_acc = accuracy_score(y_test, search_obj.predict(X_test))
        record = {
            "stage":       stage_name,
            "n_fits":      n_fits,
            "elapsed":     elapsed,
            "best_cv":     search_obj.best_score_,
            "test_acc":    test_acc,
            "best_params": search_obj.best_params_,
        }
        self.stages.append(record)
        print(f"\n  [{stage_name}]")
        print(f"    학습 횟수   : {n_fits:,} fits  ({n_fits//CV} 조합 × {CV}-fold)")
        print(f"    소요 시간   : {elapsed:.2f}s")
        print(f"    Best CV Acc : {search_obj.best_score_:.4f}")
        print(f"    Test Acc    : {test_acc:.4f}")
        print(f"    Best Params : {search_obj.best_params_}")
        return record

    def summary(self):
        total_fits    = sum(s["n_fits"] for s in self.stages)
        total_elapsed = time.time() - self._exp_start
        final_acc     = self.stages[-1]["test_acc"]
        print(f"\n  ── 실험 종료 ────────────────────────────────────────")
        print(f"    총 학습 횟수 : {total_fits:,} fits")
        print(f"    총 소요 시간 : {total_elapsed:.2f}s")
        print(f"    최종 Test Acc: {final_acc:.4f}")
        return {"total_fits": total_fits, "total_elapsed": total_elapsed, "final_acc": final_acc}

## Baseline

In [ ]:
baseline = RandomForestClassifier(random_state=SEED)
t0 = time.time()
baseline.fit(X_train, y_train)
baseline_elapsed = time.time() - t0
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))
baseline_cv  = cross_val_score(baseline, X_train, y_train, cv=CV).mean()

print(f"Baseline Test Acc : {baseline_acc:.4f}")
print(f"Baseline CV Acc   : {baseline_cv:.4f}")
print(f"Time              : {baseline_elapsed:.2f}s")

---
## Experiment A — Grid Search Only

동일한 파라미터 공간을 GridSearch 단독으로 전수 탐색합니다.
파라미터 공간이 넓을수록 조합 수가 기하급수적으로 증가합니다.

In [ ]:
# Experiment A에서 커버할 파라미터 공간
param_grid_A = {
    "n_estimators":      [50, 100, 200, 300, 500],
    "max_depth":         [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf":  [1, 2, 5, 10],
    "max_features":      ["sqrt", "log2"],
}

n_combinations_A = 1
for v in param_grid_A.values():
    n_combinations_A *= len(v)
n_fits_A = n_combinations_A * CV

print("Experiment A — Grid Search Only 파라미터 공간:")
for k, v in param_grid_A.items():
    print(f"  {k:22s}: {v}")
print(f"\n조합 수  : {n_combinations_A:,}가지")
print(f"학습 횟수 : {n_fits_A:,} fits  ({n_combinations_A:,} × {CV}-fold)")

In [ ]:
logger_A = SearchLogger("Experiment A — Grid Search Only")
logger_A.start_experiment()

grid_A = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_grid_A,
    cv=CV,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)

t0 = time.time()
grid_A.fit(X_train, y_train)
elapsed_A = time.time() - t0

logger_A.log_stage("Grid Search", grid_A, n_fits_A, elapsed_A)
summary_A = logger_A.summary()

---
## Experiment B — Random Search → Grid Search Pipeline

Stage 1에서 RandomSearch로 유망 구간을 파악하고,
Stage 2에서 해당 구간 주변을 GridSearch로 정밀 탐색합니다.

In [ ]:
def narrow_int_range(value, step, n_side=2, min_val=1):
    candidates = [value + step * i for i in range(-n_side, n_side + 1)]
    return sorted(set(max(min_val, v) for v in candidates))

logger_B = SearchLogger("Experiment B — Random + Grid Pipeline")
logger_B.start_experiment()

# ── Stage 1: RandomSearch (광역 탐색) ──
param_dist_B = {
    "n_estimators":      randint(10, 600),
    "max_depth":         [None, 5, 10, 15, 20, 30],
    "min_samples_split": randint(2, 25),
    "min_samples_leaf":  randint(1, 15),
    "max_features":      ["sqrt", "log2"],
}
N_ITER = 50

random_B = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_dist_B,
    n_iter=N_ITER,
    cv=CV,
    scoring="accuracy",
    random_state=SEED,
    n_jobs=-1,
    verbose=0,
)

t0 = time.time()
random_B.fit(X_train, y_train)
elapsed_B1 = time.time() - t0
n_fits_B1  = N_ITER * CV

stage1 = logger_B.log_stage("Stage 1: Random Search", random_B, n_fits_B1, elapsed_B1)
best = random_B.best_params_

In [ ]:
# ── Stage 2: GridSearch (정밀 탐색) ──
n_est = best["n_estimators"]
depth = best["max_depth"]
mss   = best["min_samples_split"]
msl   = best["min_samples_leaf"]
mf    = best["max_features"]

depth_grid = ([None, max(1, depth-2), depth, depth+2]
              if depth is not None else [None, 5, 10])
depth_grid = sorted(set(v for v in depth_grid if v is None or v > 0),
                    key=lambda x: (x is not None, x))

param_grid_B2 = {
    "n_estimators":      narrow_int_range(n_est, step=25, n_side=2, min_val=10),
    "max_depth":         depth_grid,
    "min_samples_split": narrow_int_range(mss,   step=1,  n_side=2, min_val=2),
    "min_samples_leaf":  narrow_int_range(msl,   step=1,  n_side=2, min_val=1),
    "max_features":      [mf],
}

n_combinations_B2 = 1
for v in param_grid_B2.values(): n_combinations_B2 *= len(v)
n_fits_B2 = n_combinations_B2 * CV

print("Stage 2 격자 (Stage 1 결과 기반 자동 생성):")
for k, v in param_grid_B2.items():
    print(f"  {k:22s}: {v}")
print(f"\n조합 수  : {n_combinations_B2}가지")
print(f"학습 횟수 : {n_fits_B2} fits")

grid_B2 = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_grid_B2,
    cv=CV,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)

t0 = time.time()
grid_B2.fit(X_train, y_train)
elapsed_B2 = time.time() - t0

stage2 = logger_B.log_stage("Stage 2: Grid Search", grid_B2, n_fits_B2, elapsed_B2)
summary_B = logger_B.summary()

---
## 최종 비교

In [ ]:
print(f"\n{'='*65}")
print(f"  FINAL COMPARISON")
print(f"{'='*65}")
print(f"  {'':30s}  {'Baseline':>10}  {'Exp A':>10}  {'Exp B':>10}")
print(f"  {'-'*60}")

# 정확도
print(f"  {'Test Accuracy':30s}  "
      f"{baseline_acc:>10.4f}  "
      f"{summary_A['final_acc']:>10.4f}  "
      f"{summary_B['final_acc']:>10.4f}")

# 총 학습 횟수
print(f"  {'Total Fits':30s}  "
      f"{'1':>10}  "
      f"{summary_A['total_fits']:>10,}  "
      f"{summary_B['total_fits']:>10,}")

# 소요 시간
print(f"  {'Total Time (s)':30s}  "
      f"{baseline_elapsed:>10.2f}  "
      f"{summary_A['total_elapsed']:>10.2f}  "
      f"{summary_B['total_elapsed']:>10.2f}")

# 효율 (정확도 향상 / 학습 횟수)
gain_A = (summary_A['final_acc'] - baseline_acc) * 100
gain_B = (summary_B['final_acc'] - baseline_acc) * 100
eff_A  = gain_A / summary_A['total_fits'] * 1000
eff_B  = gain_B / summary_B['total_fits'] * 1000

print(f"\n  Baseline 대비 정확도 향상")
print(f"    Exp A: {gain_A:+.2f}%p  ({summary_A['total_fits']:,} fits)")
print(f"    Exp B: {gain_B:+.2f}%p  ({summary_B['total_fits']:,} fits)")
print(f"\n  효율 (향상%p / 1000 fits)")
print(f"    Exp A: {eff_A:.4f}")
print(f"    Exp B: {eff_B:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
labels = ["Baseline", "Exp A\n(Grid Only)", "Exp B\n(Random+Grid)"]
colors = ["#aaaaaa", "steelblue", "coral"]

# 정확도
accs = [baseline_acc, summary_A["final_acc"], summary_B["final_acc"]]
axes[0].bar(labels, accs, color=colors, alpha=0.85, edgecolor="white")
axes[0].set_title("Test Accuracy"); axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(min(accs) - 0.03, 1.0)
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.002, f"{v:.4f}", ha="center", fontsize=10)
axes[0].grid(axis="y", alpha=0.3)

# 총 학습 횟수
fits = [1, summary_A["total_fits"], summary_B["total_fits"]]
axes[1].bar(labels, fits, color=colors, alpha=0.85, edgecolor="white")
axes[1].set_title("Total Fits (학습 횟수)"); axes[1].set_ylabel("# Fits")
for i, v in enumerate(fits):
    axes[1].text(i, v + max(fits)*0.01, f"{v:,}", ha="center", fontsize=9)
axes[1].grid(axis="y", alpha=0.3)

# 소요 시간
times = [baseline_elapsed, summary_A["total_elapsed"], summary_B["total_elapsed"]]
axes[2].bar(labels, times, color=colors, alpha=0.85, edgecolor="white")
axes[2].set_title("Total Time (s)"); axes[2].set_ylabel("Seconds")
for i, v in enumerate(times):
    axes[2].text(i, v + max(times)*0.01, f"{v:.1f}s", ha="center", fontsize=10)
axes[2].grid(axis="y", alpha=0.3)

plt.suptitle("Grid Only vs Random+Grid Pipeline", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 정리

```
Experiment A (Grid Only):
  [넓은 파라미터 공간] ──→ GridSearch 전수 탐색 ──→ Best Params
                              (대량의 fits, 긴 시간)

Experiment B (Random+Grid):
  [넓은 파라미터 공간] ──→ RandomSearch (50 fits)
                                    ↓
                          [유망 구간 자동 추출]
                                    ↓
                          GridSearch 정밀 탐색 ──→ Best Params
                              (소량의 fits, 짧은 시간)
```

**핵심 관찰**: Experiment B는 Experiment A보다 훨씬 적은 학습 횟수로 동등하거나 더 나은 정확도를 달성한다.
파라미터 공간이 넓을수록 이 차이는 더욱 극명해진다.